# Lab 1.1, Build 2: Repair the Prompt

**Run this first:** select **Cell > Run All** to restore the harness and review your Build 1 diagnoses.

Cells marked `# ── YOUR WORK ──` are the ones you edit.

In [ ]:
# ── Run this first — no edits needed ──────────────────────────────────────────
# Re-initializes the harness and prints your Build 1 diagnoses for reference.
import sys, os, json, pathlib, time

sys.path.insert(0, '/opt/ara/lib')
from tina.client import llm_client, model_fast, model_strong

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

client = llm_client()
FAST   = model_fast()
STRONG = model_strong()
TRACES = pathlib.Path('/home/elastic/.traces')
TRACES.mkdir(parents=True, exist_ok=True)
RESULTS = pathlib.Path('/opt/ara/results')

print(f'Harness ready. FAST={FAST}, STRONG={STRONG}')

# Print Build 1 diagnoses for reference
diag_path = TRACES / 'diagnose-results.json'
if diag_path.exists():
    diag = json.loads(diag_path.read_text())
    print('\nBuild 1 diagnoses (from lab-1-1-triage.ipynb):')
    for i, d in enumerate(diag.get('diagnoses', []), 1):
        print(f'  Response {i}: failure_mode={d["failure_mode"]!r}, model_type={d["model_type"]!r}')
else:
    print('\nNo Build 1 diagnoses found — complete lab-1-1-triage.ipynb and select Check first.')

---
## Your new work starts here

## Build 2: Repair the system prompt

The broken prompt below produces the ungrounded responses you saw. Rewrite it so Tina's answers are schema-valid AND grounded in retrieved policy values.

Mark each section with `[ROLE]`, `[SCHEMA]`, `[CONSTRAINTS]`, `[GROUNDING]` so the ablation check can identify which element was decisive.

In [ ]:
# ── The broken system prompt (reference only — do not edit this cell) ─────────
BROKEN_PROMPT = '''You are a helpful assistant. Answer the user's question.
Format your response as JSON with these fields: answer, policy_id, confidence.'''

print('Broken prompt:')
print(BROKEN_PROMPT)

In [ ]:
# ── Test your prompt on the dev set ─────────────────────────────────────────
# Run this cell as often as you like while iterating.
DEV_PROMPT = BROKEN_PROMPT  # ← replace with your rewritten prompt

dev_q_path = pathlib.Path('/home/elastic/dev-sets/dev-questions.jsonl')
if dev_q_path.exists():
    questions = [json.loads(l) for l in dev_q_path.read_text().splitlines() if l.strip()]
    schema_ok = 0; grounded = 0
    for q in questions:
        r = client.chat.completions.create(
            model=FAST,
            messages=[{'role': 'system', 'content': DEV_PROMPT},
                      {'role': 'user',   'content': q['question']}],
            temperature=0,
        )
        ans = r.choices[0].message.content or ''
        try:
            parsed = json.loads(ans)
            sv = all(f in parsed for f in ['answer', 'policy_id', 'confidence'])
        except Exception:
            sv = False
        gr = any(kw.lower() in ans.lower() for kw in q.get('gold_keywords', []))
        schema_ok += 1 if sv else 0; grounded += 1 if gr else 0
        print(f"  {q['id']}: schema={'OK' if sv else 'FAIL'}, grounded={'OK' if gr else 'FAIL'}")
    print(f'\nDev set: {schema_ok}/4 schema-valid, {grounded}/4 grounded')

In [ ]:
# ── YOUR WORK ── Rewrite the system prompt ───────────────────────────────────
# Include all four sections marked [ROLE], [SCHEMA], [CONSTRAINTS], [GROUNDING].

MY_SYSTEM_PROMPT = '''
[ROLE]
... your role definition here ...

[SCHEMA]
... your output schema here ...

[CONSTRAINTS]
... your constraints here ...

[GROUNDING]
... your grounding instruction here ...
'''

In [ ]:
# Save your prompt (run after you finish editing MY_SYSTEM_PROMPT above)
assert MY_SYSTEM_PROMPT.strip() and '[GROUNDING]' in MY_SYSTEM_PROMPT, \
    'Replace the placeholder text and include all four [SECTION] markers.'
pathlib.Path('/home/elastic/my_system_prompt.txt').write_text(MY_SYSTEM_PROMPT)
print('Prompt saved. Select Check in the sidebar.')